# ⏰ 02_cleaned_to_processed_eda: Feature Discovery & Hypotheses
Now that the data is clean, we look for patterns that can be turned into **predictive features**. This notebook justifies the logic inside `feature_pipeline.py`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import pathlib

# Add project root to sys.path to allow importing from src
sys.path.append("../") 
from src.data.ingest import load_raw
from src.data.validate import validate_data

# Set visualization style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Load and Validate
df_raw = load_raw(sample_mode=True, sample_size=100_000)
df, _ = validate_data(df_raw)
df['target'] = df['trip_duration']

print(f"Cleaned dataset loaded. Shape: {df.shape}")
df.head()

## 1. Temporal Patterns
Does the time of day affect the trip duration? We analyze the "Rush Hour" hypothesis.

In [ ]:
df['hour'] = pd.to_datetime(df['pickup_datetime']).dt.hour
plt.figure(figsize=(12, 5))
sns.lineplot(data=df, x='hour', y='target', estimator='mean')
plt.title("Average Trip Duration by Hour of Day")
plt.xlabel("Hour of Day (0-23)")
plt.ylabel("Mean Duration (Seconds)")
plt.show()

## 2. Day-of-Week Analysis
Testing if the "Weekend vs Weekday" behavior is significant.

In [ ]:
df['day_of_week'] = pd.to_datetime(df['pickup_datetime']).dt.dayofweek
plt.figure(figsize=(10, 5))
sns.boxplot(data=df, x='day_of_week', y='target')
plt.title("Trip Duration by Day of Week (0=Monday)")
plt.xlabel("Day of Week")
plt.ylabel("Duration (Seconds)")
plt.show()

### ✍️ Engineering Inferences:
1. **Cyclical Time:** The hour plot shows clear peaks and troughs $\rightarrow$ Justifies the use of `hour_sin` and `hour_cos` transformations.
2. **Daily Rhythm:** The boxplots for `day_of_week` show varying medians and spreads $\rightarrow$ Justifies including `day_of_week` as a categorical feature.
3. **Rush Hour:** The spikes in the line plot identify specific time windows that we can capture with a binary `is_rush_hour` feature.